In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'gmlp_mcmc/training'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


# Train gMLP DeepRV On The 760 Soay Locations

The CSV `z` column supplies the fixed GP locations `s` in their original row order. DeepRV separately draws a 760-dimensional standard-normal latent input for each simulated RBF GP realization.

In [ ]:
import sys
from pathlib import Path

repo_root = PROJECT_ROOT
sys.path.insert(0, str(repo_root))

from jax import random
from dl4bi.vae.train_utils import generate_surrogate_decoder
from experiments.gmlp_mcmc.train_gmlp_deeprv_soay_760 import TrainConfig, load_checkpoint, load_locations, output_paths, train_model

In [ ]:
config = TrainConfig(data_path=str(GENERATED / 'soay/size_only/data/gp_ibm_dataset.csv'), output_dir=str(GENERATED / 'gmlp_mcmc/training'), batch_size=32, num_steps=200_000, checkpoint_interval=25_000, validation_interval=25_000, validation_steps=5_000, learning_rate=1e-3, weight_decay=1e-2, clip_norm=3.0, num_blocks=2, wandb_mode="disabled")
locations = load_locations(config)
config, locations.shape, (float(locations.min()), float(locations.max())), output_paths(config)

`latest_soay_760.ckpt` is overwritten every 25,000 steps. `model_soay_760.ckpt` stores the best validation state, `final_soay_760.ckpt` stores the final state, and `metadata_soay_760.json` records the dataset fingerprint and training results.

In [ ]:
result = train_model(config)
result["metadata"]

Load the best-validation checkpoint and create the original DeepRV decoder. The loader remaps a checkpoint trained on a GPU to the current CPU or GPU.

In [ ]:
model, best_state = load_checkpoint(output_paths(config)["model"], config)
decoder = generate_surrogate_decoder(best_state, model)
epsilon = random.normal(random.key(42), (1, config.num_locations))
draw = decoder(epsilon, random.uniform(random.key(43), (1,), minval=config.lengthscale_min, maxval=config.lengthscale_max), s=locations)
int(best_state.step), draw.shape